# 🏦 Customer Support Handoff Orchestration

## Overview

This notebook demonstrates **Handoff orchestration** for customer support — a multi-agent pattern where agents dynamically transfer control to one another based on context or user request. Each agent can "handoff" the conversation to another agent with appropriate expertise, ensuring the right specialist handles each part of the task.

Internally, the handoff orchestration is implemented using a **mesh topology** where agents are connected directly without an orchestrator. Each agent can decide when to hand off the conversation based on predefined rules or the content of the messages.

> ⚠️ **Note**: Handoff orchestration only supports `Agent` and the agents must support local tools execution.

### 💼 Industry Use Case: Customer Support with Dynamic Agent Routing

A financial services company uses specialized agents for different support areas:
1. **Triage Agent** receives the initial customer inquiry and routes it
2. **Order Agent** handles order tracking and shipping inquiries
3. **Return Agent** manages product return requests
4. **Refund Agent** processes refund requests

### Key Concepts

| Concept | Description |
|---------|-------------|
| **HandoffBuilder** | High-level API for building handoff workflows |
| **Dynamic Routing** | Agents decide which agent handles the next interaction |
| **with_start_agent()** | Defines which agent receives user input first |
| **add_handoff()** | Configures specific handoff relationships between agents |
| **HandoffAgentUserRequest** | Event emitted when an agent needs user input |
| **Context Synchronization** | Full conversation history maintained across handoffs |
| **Autonomous Mode** | Optional mode that bypasses human input requirements |
| **Tool Approval (HITL)** | Human approval for sensitive tool operations |
| **Checkpointing** | Durable workflows that survive process restarts |

### Differences Between Handoff and Agent-as-Tools

| Aspect | Handoff | Agent-as-Tools |
|--------|---------|----------------|
| **Control Flow** | Explicitly passed between agents; no central authority | Primary agent delegates subtasks; control always returns |
| **Task Ownership** | Receiving agent takes full ownership | Primary agent retains overall responsibility |
| **Context** | Full context handed off entirely | Primary agent manages context selectively |

### Architecture

```
Customer Support Request
           ↓
    Triage Agent (routes inquiry)
           ↓
┌──────────────────────────────────┐
│   Dynamic Handoff Routing        │
│  ├── Order Agent (tracking)      │
│  ├── Return Agent (returns)      │
│  └── Refund Agent (refunds)      │
│       ↕ agents can handoff back  │
│         to triage or each other  │
└──────────────────────────────────┘
           ↓
    Resolution & Workflow Complete
```

### What You'll Learn

- How to create specialized agents for different domains
- How to configure handoff rules between agents
- How to build interactive workflows with dynamic agent routing
- How to handle multi-turn conversations with agent switching
- How to implement tool approval for sensitive operations (HITL)
- How to use checkpointing for durable handoff workflows

## Prerequisites

- ✅ Azure AI Foundry project configured
- ✅ Environment variables: `AI_FOUNDRY_PROJECT_ENDPOINT`, `AZURE_AI_MODEL_DEPLOYMENT_NAME`
- ✅ Azure CLI authentication: Run `az login`

## 1️⃣ Install Dependencies and Import Libraries

In [ ]:
# Install required packages (uncomment if needed)
# %pip install agent-framework azure-identity python-dotenv

import asyncio
import os
from typing import Annotated
from dataclasses import dataclass
from dotenv import load_dotenv

from azure.identity import AzureCliCredential

from agent_framework import (
    Agent,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Content,
    Executor,
    FileCheckpointStorage,
    Message,
    tool,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowEvent,
    WorkflowRunState,
    handler,
    response_handler,
)
from agent_framework.foundry import FoundryChatClient

# Load environment variables from .env file
load_dotenv('../../.env')

# Verify environment is loaded
PROJECT_ENDPOINT = os.getenv("AI_FOUNDRY_PROJECT_ENDPOINT")
MODEL_DEPLOYMENT = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")
print(f"✅ Environment loaded: {PROJECT_ENDPOINT is not None and MODEL_DEPLOYMENT is not None}")

## 2️⃣ Define Simulated Tools for Customer Support

Define three tool-decorated functions that the specialist agents will use. Each tool simulates a real customer support operation.

In [ ]:
@tool
def process_refund(order_number: Annotated[str, "Order number to process refund for"]) -> str:
    """Simulated function to process a refund for a given order number."""
    return f"Refund processed successfully for order {order_number}."

@tool
def check_order_status(order_number: Annotated[str, "Order number to check status for"]) -> str:
    """Simulated function to check the status of a given order number."""
    return f"Order {order_number} is currently being processed and will ship in 2 business days."

@tool
def process_return(order_number: Annotated[str, "Order number to process return for"]) -> str:
    """Simulated function to process a return for a given order number."""
    return f"Return initiated successfully for order {order_number}. You will receive return instructions via email."

print("✅ Customer support tools defined: process_refund, check_order_status, process_return")

✅ Customer support tools defined: process_refund, check_order_status, process_return


## 3️⃣ Set Up the Foundry Chat Client

Initialize the `FoundryChatClient` using `AzureCliCredential` for authentication. This client will be used to create all agents.

In [ ]:
# Create shared Foundry Chat Client
chat_client = FoundryChatClient(
    credential=AzureCliCredential(),
    project_endpoint=PROJECT_ENDPOINT,
    model=MODEL_DEPLOYMENT,
)

print("✅ Foundry Chat Client created")

## 4️⃣ Define Specialized Customer Support Agents

Create domain-specific agents with a triage coordinator for routing.

### Agent Roles

| Agent | Role | Tools |
|-------|------|-------|
| **Triage Agent** | Routes customer inquiries to the right specialist | None |
| **Order Agent** | Handles order tracking and shipping issues | `check_order_status` |
| **Return Agent** | Manages product return requests | `process_return` |
| **Refund Agent** | Processes refund requests | `process_refund` |

In [ ]:
# Create triage/coordinator agent
triage_agent = Agent(
    client=chat_client,
    instructions=(
        "You are frontline support triage at a financial services company. "
        "Route customer issues to the appropriate specialist agents based on the problem described. "
        "Ask clarifying questions if the issue is unclear before routing."
    ),
    description="Triage agent that handles general inquiries and routes to specialists.",
    name="triage_agent",
)

# Refund specialist: Handles refund requests
refund_agent = Agent(
    client=chat_client,
    instructions=(
        "You process refund requests for a financial services company. "
        "Always verify the order number before processing. "
        "Explain the refund timeline and process to the customer."
    ),
    description="Agent that handles refund requests.",
    name="refund_agent",
    tools=[process_refund],
)

# Order/shipping specialist: Resolves delivery issues
order_agent = Agent(
    client=chat_client,
    instructions=(
        "You handle order and shipping inquiries for a financial services company. "
        "Provide clear status updates and estimated delivery timelines."
    ),
    description="Agent that handles order tracking and shipping issues.",
    name="order_agent",
    tools=[check_order_status],
)

# Return specialist: Handles return requests
return_agent = Agent(
    client=chat_client,
    instructions=(
        "You manage product return requests for a financial services company. "
        "Guide customers through the return process step by step. "
        "If a customer also wants a refund after the return, route them to the refund specialist."
    ),
    description="Agent that handles return processing.",
    name="return_agent",
    tools=[process_return],
)

print("✅ Triage Agent created")
print("✅ Refund Agent created")
print("✅ Order Agent created")
print("✅ Return Agent created")

## 5️⃣ Configure Basic Handoff Workflow with HandoffBuilder

By default, all agents can handoff to each other. The `HandoffBuilder` automatically registers handoff tools with each agent.

| Method | Purpose |
|--------|---------|
| `HandoffBuilder()` | Initialize the handoff orchestration builder |
| `.with_start_agent()` | Define which agent receives user input first |
| `termination_condition` | Lambda to determine when the workflow should stop |
| `.build()` | Create the workflow |

In [ ]:
print("\n🏦 Building Customer Support Handoff Workflow...")

@dataclass
class HandoffRequest:
    """Request for the coordinator to route to a specific agent."""
    target_agent: str
    user_message: str
    conversation_history: list[Message]

@dataclass 
class UserInputRequest:
    """Request for user input when agent needs more information."""
    agent_name: str
    agent_message: str

class HandoffCoordinator(Executor):
    """Coordinates handoffs between customer support agents.
    
    Routes messages to the appropriate specialist agent and manages
    the conversation flow, including requesting user input when needed.
    """
    
    def __init__(self, id: str, agents: dict[str, Agent]):
        super().__init__(id=id)
        self._agents = agents
        self._conversation: list[Message] = []
        self._current_agent: str = "triage_agent"
    
    @handler
    async def start(self, user_message: str, ctx: WorkflowContext) -> None:
        """Handle initial user message by routing to triage agent."""
        self._conversation.append(Message(role="user", text=user_message))
        await self._route_to_agent(self._current_agent, ctx)
    
    async def _route_to_agent(self, agent_name: str, ctx: WorkflowContext) -> None:
        """Route the conversation to the specified agent."""
        self._current_agent = agent_name
        agent = self._agents[agent_name]
        
        print(f"\n🔄 Routing to {agent_name}...")
        
        # Add routing context to help agent decide
        routing_msg = Message(
            role="system",
            text=(
                f"You are now handling this conversation. "
                f"Available agents to handoff to: {', '.join(self._agents.keys())}. "
                f"If you need to route to another agent, say 'HANDOFF TO: <agent_name>' at the start of your response. "
                f"If the conversation is resolved, say 'RESOLVED' at the start."
            )
        )
        
        messages_for_agent = [routing_msg] + self._conversation
        response = await agent.run(messages_for_agent)
        response_text = response.text or ""
        
        self._conversation.append(Message(role="assistant", text=response_text, author_name=agent_name))
        
        # Check if agent wants to handoff
        if response_text.upper().startswith("HANDOFF TO:"):
            target = response_text.split(":", 1)[1].strip().split("\n")[0].strip().lower()
            # Find matching agent
            for name in self._agents:
                if name.lower() == target or target in name.lower():
                    await self._route_to_agent(name, ctx)
                    return
        
        # Check if resolved
        if response_text.upper().startswith("RESOLVED") or "welcome" in response_text.lower():
            await ctx.yield_output(f"Conversation resolved by {agent_name}:\n{response_text}")
            return
        
        # Agent responded but needs user input - request it
        await ctx.request_info(
            request_data=UserInputRequest(agent_name=agent_name, agent_message=response_text),
            response_type=str,
        )
    
    @response_handler
    async def on_user_input(
        self,
        original_request: UserInputRequest,
        user_input: str,
        ctx: WorkflowContext,
    ) -> None:
        """Handle user's response and continue the conversation."""
        self._conversation.append(Message(role="user", text=user_input))
        await self._route_to_agent(self._current_agent, ctx)


# Build the handoff workflow
coordinator = HandoffCoordinator(
    id="coordinator",
    agents={
        "triage_agent": triage_agent,
        "order_agent": order_agent,
        "return_agent": return_agent,
        "refund_agent": refund_agent,
    }
)

workflow = WorkflowBuilder(start_executor=coordinator).build()

print("✅ Handoff workflow built - coordinator manages agent routing")


🏦 Building Customer Support Handoff Workflow (Default Routing)...
✅ Handoff workflow built - all agents can handoff to each other


## 6️⃣ Configure Advanced Handoff Rules (Custom Routing)

For more advanced routing, you can configure specific handoff relationships between agents using `add_handoff()`.

### Custom Routing Rules
```
triage_agent ──→ order_agent, return_agent  (cannot route directly to refund)
return_agent ──→ refund_agent, triage_agent (returns can escalate to refunds)
order_agent  ──→ triage_agent              (orders route back to triage)
refund_agent ──→ triage_agent              (refunds route back to triage)
```

> **Note:** Even with custom handoff rules, all agents are still connected in a mesh topology for **context synchronization**. The handoff rules only govern which agents can **take over** the conversation next. Only user and agent messages are synchronized — tool-related contents (including handoff tool calls) are not broadcasted.

In [ ]:
print("\n🏦 Building Customer Support Handoff Workflow (Custom Routing)...")

# Custom routing rules: restrict which agents can handoff to which
CUSTOM_ROUTING_RULES = {
    "triage_agent": ["order_agent", "return_agent"],  # Cannot route directly to refund
    "return_agent": ["refund_agent", "triage_agent"],  # Returns can escalate to refunds
    "order_agent": ["triage_agent"],  # Orders route back to triage
    "refund_agent": ["triage_agent"],  # Refunds route back to triage
}

class HandoffCoordinatorCustom(HandoffCoordinator):
    """Coordinator with custom routing rules."""
    
    def __init__(self, id: str, agents: dict[str, Agent], routing_rules: dict[str, list[str]]):
        super().__init__(id=id, agents=agents)
        self._routing_rules = routing_rules
    
    async def _route_to_agent(self, agent_name: str, ctx: WorkflowContext) -> None:
        """Route with custom rules about allowed handoff targets."""
        self._current_agent = agent_name
        agent = self._agents[agent_name]
        
        allowed_targets = self._routing_rules.get(agent_name, list(self._agents.keys()))
        
        print(f"\n🔄 Routing to {agent_name} (can handoff to: {', '.join(allowed_targets)})...")
        
        routing_msg = Message(
            role="system",
            text=(
                f"You are now handling this conversation. "
                f"You can ONLY handoff to these agents: {', '.join(allowed_targets)}. "
                f"To handoff, say 'HANDOFF TO: <agent_name>' at the start. "
                f"If resolved, say 'RESOLVED' at the start."
            )
        )
        
        messages_for_agent = [routing_msg] + self._conversation
        response = await agent.run(messages_for_agent)
        response_text = response.text or ""
        
        self._conversation.append(Message(role="assistant", text=response_text, author_name=agent_name))
        
        if response_text.upper().startswith("HANDOFF TO:"):
            target = response_text.split(":", 1)[1].strip().split("\n")[0].strip().lower()
            for name in allowed_targets:
                if name.lower() == target or target in name.lower():
                    await self._route_to_agent(name, ctx)
                    return
        
        if response_text.upper().startswith("RESOLVED") or "welcome" in response_text.lower():
            await ctx.yield_output(f"Conversation resolved by {agent_name}:\n{response_text}")
            return
        
        await ctx.request_info(
            request_data=UserInputRequest(agent_name=agent_name, agent_message=response_text),
            response_type=str,
        )

coordinator_custom = HandoffCoordinatorCustom(
    id="coordinator_custom",
    agents={
        "triage_agent": triage_agent,
        "order_agent": order_agent,
        "return_agent": return_agent,
        "refund_agent": refund_agent,
    },
    routing_rules=CUSTOM_ROUTING_RULES,
)

workflow_custom = WorkflowBuilder(start_executor=coordinator_custom).build()

print("✅ Custom routing handoff workflow built")
print("   triage → order, return")
print("   return → refund, triage")
print("   order  → triage")
print("   refund → triage")


🏦 Building Customer Support Handoff Workflow (Custom Routing)...
✅ Custom routing handoff workflow built
   triage → order, return
   return → refund, triage
   order  → triage
   refund → triage


## 7️⃣ Run Interactive Handoff Agent Interaction

Unlike other orchestrations, handoff is **interactive** because an agent may not decide to handoff after every turn. If an agent doesn't handoff, human input is required to continue the conversation.

### Interaction Flow
```
User Message → Agent Processes →
  ├── Agent calls handoff tool → Next agent takes over
  └── Agent responds directly  → Workflow pauses with RequestInfoEvent
```

When an agent decides not to handoff, the workflow emits a `RequestInfoEvent` with a `HandoffAgentUserRequest` payload containing the agent's most recent messages. Your application then collects the user's input and calls `workflow.send_responses()` to resume.

> **Note:** Each run cell builds a **fresh workflow** to avoid stale state from previous interrupted runs in the notebook.

In [ ]:
# Build a fresh workflow for the interactive demo
coordinator_interactive = HandoffCoordinator(
    id="coordinator_interactive",
    agents={
        "triage_agent": triage_agent,
        "order_agent": order_agent,
        "return_agent": return_agent,
        "refund_agent": refund_agent,
    }
)
workflow_interactive = WorkflowBuilder(start_executor=coordinator_interactive).build()

# Interactive handoff workflow
pending_responses: dict[str, object] | None = None
completed = False

print("🚀 Starting interactive handoff workflow...")
print("=" * 60)

while not completed:
    # First iteration: start with user message
    # Subsequent iterations: send responses to pending requests
    if pending_responses:
        result = await workflow_interactive.run(responses=pending_responses)
    else:
        result = await workflow_interactive.run("I need help with my order")
    pending_responses = None

    # Check for pending user-input requests
    requests: list[tuple[str, UserInputRequest]] = []
    for event in result.get_request_info_events():
        if isinstance(event.data, UserInputRequest):
            requests.append((event.request_id, event.data))

    # Check for workflow output (completion)
    outputs = result.get_outputs()
    if outputs:
        completed = True

    # Check if workflow is idle with pending requests
    final_state = result.get_final_state()
    if final_state == WorkflowRunState.IDLE_WITH_PENDING_REQUESTS:
        print("\n⏳ Workflow paused — awaiting your input...")

    if completed:
        print("\n✅ Workflow completed!")
        if outputs:
            print(outputs[0])
        break

    # Collect user input for each pending request
    if requests:
        pending_responses = {}
        for request_id, req_data in requests:
            print(f"\n💬 Agent '{req_data.agent_name}' says:")
            print(f"   {req_data.agent_message}")
            print("-" * 40)
            user_input = input("You: ").strip()
            if user_input.lower() == "exit":
                print("\n❌ Conversation ended by user.")
                completed = True
                break
            pending_responses[request_id] = user_input
    else:
        print("\n✅ No more pending requests. Workflow complete.")
        break

print("\n✅ Interactive handoff demo complete!")

🚀 Starting interactive handoff workflow...

⏳ Workflow paused — awaiting your input...

💬 Agent 'triage_agent' says:
   triage_agent: Could you please provide more details about your order issue? For example, are you trying to track your order, having trouble with the shipping, or is it related to something else?
----------------------------------------

⏳ Workflow paused — awaiting your input...

💬 Agent 'triage_agent' says:
   triage_agent: Could you clarify the problem with your order?
----------------------------------------

⏳ Workflow paused — awaiting your input...

💬 Agent 'triage_agent' says:
   triage_agent: If you can provide more details about your order issue, I'll connect you to the right specialist.
----------------------------------------

⏳ Workflow paused — awaiting your input...

💬 Agent 'return_agent' says:
   return_agent: If you'd like to process a return for your order, kindly share your order number, and I'll guide you through the return process.
---------------

## 8️⃣ Enable Autonomous Mode for Non-Interactive Workflows

Handoff orchestration is designed for interactive scenarios. However, you can enable **autonomous mode** to allow the workflow to continue without human intervention. When an agent decides not to handoff, the workflow automatically sends a default response.

> **Why is Handoff orchestration inherently interactive?** Unlike other orchestrations where there is only one path to follow after an agent responds, in a Handoff orchestration the agent has the option to either handoff to another agent or continue assisting the user. Because handoffs are achieved through tool calls, if an agent generates a response instead, the workflow delegates back to the user.

| Configuration | Description |
|--------------|-------------|
| `with_autonomous_mode()` | Enable for all agents |
| `with_autonomous_mode(agents=[...])` | Enable for specific agents only |
| `prompts={...}` | Customize the autonomous response message |
| `turn_limits={...}` | Limit autonomous turns per agent |

In [ ]:
print("\n🏦 Running Autonomous Handoff Workflow...")

# For autonomous mode, we simply run the workflow without interactive input
# The coordinator will route between agents automatically
coordinator_auto = HandoffCoordinator(
    id="coordinator_auto",
    agents={
        "triage_agent": triage_agent,
        "order_agent": order_agent,
        "return_agent": return_agent,
        "refund_agent": refund_agent,
    }
)
workflow_autonomous = WorkflowBuilder(start_executor=coordinator_auto).build()

# Run - the coordinator handles routing automatically
result = await workflow_autonomous.run("Check the status of order ORD-12345")

outputs = result.get_outputs()
if outputs:
    print(f"\n✅ Autonomous workflow completed!")
    print(outputs[0])

# If there are pending requests, show them
for event in result.get_request_info_events():
    if isinstance(event.data, UserInputRequest):
        print(f"\n💬 Agent '{event.data.agent_name}' responded:")
        print(f"   {event.data.agent_message}")


🏦 Building Autonomous Handoff Workflow...
✅ Fully autonomous handoff workflow built (all agents)

💬 Agent 'triage_agent' responded:
   triage_agent: It seems the user inquiry has already been resolved. No additional action is necessary unless any new concerns arise. Thank you!


## 9️⃣ Customize Autonomous Mode (Subset of Agents, Prompts, Turn Limits)

You can fine-tune autonomous mode with three configuration options:

1. **Subset of agents** — Only specific agents run autonomously
2. **Custom prompts** — Customize the default response message per agent
3. **Turn limits** — Cap autonomous turns to prevent runaway workflows

In [ ]:
# The HandoffCoordinator pattern handles routing variations through 
# custom routing rules (shown in the custom routing section above).
# Here we demonstrate running the custom routing workflow.

coordinator_variation = HandoffCoordinatorCustom(
    id="coordinator_variation",
    agents={
        "triage_agent": triage_agent,
        "order_agent": order_agent,
        "return_agent": return_agent,
        "refund_agent": refund_agent,
    },
    routing_rules=CUSTOM_ROUTING_RULES,
)
workflow_variation = WorkflowBuilder(start_executor=coordinator_variation).build()

result = await workflow_variation.run("I received a damaged item and want to return it for a refund")

outputs = result.get_outputs()
if outputs:
    print(f"\n✅ Custom routing workflow completed!")
    print(outputs[0])

for event in result.get_request_info_events():
    if isinstance(event.data, UserInputRequest):
        print(f"\n💬 Agent '{event.data.agent_name}' says:")
        print(f"   {event.data.agent_message}")

## 🔟 Advanced: Define Tools with Approval Required (HITL)

Handoff workflows can include agents with tools that require **human approval** before execution. This is useful for sensitive operations like processing refunds or executing irreversible actions.

Use `@ai_function(approval_mode="always_require")` to mark a tool as requiring approval.

In [ ]:
# Redefine process_refund with approval required
@tool(approval_mode="always_require")
def process_refund_with_approval(order_number: Annotated[str, "Order number to process refund for"]) -> str:
    """Simulated function to process a refund — requires human approval before execution."""
    return f"Refund processed successfully for order {order_number}."

# Recreate agents with the approval-required tool
triage_agent_hitl = Agent(
    client=chat_client,
    instructions=(
        "You are frontline support triage. Route customer issues to the appropriate specialist agents "
        "based on the problem described."
    ),
    description="Triage agent that handles general inquiries.",
    name="triage_agent",
)

refund_agent_hitl = Agent(
    client=chat_client,
    instructions="You process refund requests. Always verify the order number before processing.",
    description="Agent that handles refund requests (requires approval).",
    name="refund_agent",
    tools=[process_refund_with_approval],  # This tool requires human approval
)

order_agent_hitl = Agent(
    client=chat_client,
    instructions="You handle order and shipping inquiries.",
    description="Agent that handles order tracking and shipping issues.",
    name="order_agent",
    tools=[check_order_status],
)

print("✅ Agents with approval-required tools created")
print("   refund_agent now requires human approval for process_refund")

## 1️⃣1️⃣ Handle Both User Input and Tool Approval Requests

When running a handoff workflow with approval-required tools, pending requests can be either:
- **`HandoffAgentUserRequest`** — Agent needs user input to continue
- **`FunctionApprovalRequestContent`** — Agent wants to call a tool that requires human approval

You must check the type of each request and respond appropriately.

In [ ]:
# Build workflow with approval-required tools
coordinator_hitl = HandoffCoordinator(
    id="coordinator_hitl",
    agents={
        "triage_agent": triage_agent_hitl,
        "order_agent": order_agent_hitl,
        "refund_agent": refund_agent_hitl,
    }
)

workflow_with_approvals = WorkflowBuilder(start_executor=coordinator_hitl).build()

print("✅ Handoff workflow with tool approval built")

# Run the workflow handling both user input and tool approval requests
pending_responses: dict[str, object] | None = None
completed = False

while not completed:
    if pending_responses:
        result = await workflow_with_approvals.run(responses=pending_responses)
    else:
        result = await workflow_with_approvals.run("My order 12345 arrived damaged. I need a refund.")
    pending_responses = None

    # Check for completion
    outputs = result.get_outputs()
    if outputs:
        print(f"\n✅ Workflow completed!")
        print(outputs[0])
        break

    # Process pending requests
    info_events = result.get_request_info_events()
    if not info_events:
        print("\n✅ No more pending requests. Workflow complete.")
        break

    pending_responses = {}
    for request in info_events:
        if isinstance(request.data, UserInputRequest):
            # Agent needs user input
            print(f"\n💬 Agent '{request.data.agent_name}' says:")
            print(f"   {request.data.agent_message}")

            user_input = input("You: ")
            pending_responses[request.request_id] = user_input

        elif hasattr(request.data, 'function_call'):
            # Tool approval request
            func_call = request.data.function_call
            args = func_call.parse_arguments() if hasattr(func_call, 'parse_arguments') else {}

            print(f"\n🔐 Tool approval requested: {func_call.name}")
            print(f"   Arguments: {args}")

            approval = input("   Approve? (y/n): ").strip().lower() == "y"
            pending_responses[request.request_id] = request.data.to_function_approval_response(approved=approval)

## 1️⃣2️⃣ Implement Checkpointing for Durable Handoff Workflows

For long-running workflows where tool approvals may happen hours or days later, use **checkpointing** to persist workflow state. The workflow can be paused when approval is needed and resumed later from a saved checkpoint.

### Checkpoint Workflow
```
Initial Run → Workflow pauses (approval needed) → Checkpoint saved
     ...time passes...
Resume → Restore checkpoint → Send approval → Workflow completes
```

In [ ]:
from agent_framework import FileCheckpointStorage

# Create checkpoint storage
storage = FileCheckpointStorage(storage_path="./checkpoints")

# Build a durable workflow with checkpoint storage
coordinator_durable = HandoffCoordinator(
    id="coordinator_durable",
    agents={
        "triage_agent": triage_agent_hitl,
        "order_agent": order_agent_hitl,
        "refund_agent": refund_agent_hitl,
    }
)

workflow_durable = WorkflowBuilder(
    start_executor=coordinator_durable,
    checkpoint_storage=storage,
).build()

print("✅ Durable handoff workflow with checkpointing built")

# --- Phase 1: Initial run — workflow pauses when user input is needed ---
print("\n📌 Phase 1: Starting workflow (will pause for user input)...")
result = await workflow_durable.run("I need a refund for order 12345")
pending = result.get_request_info_events()
print(f"   {len(pending)} pending request(s) saved. Checkpoint stored automatically.")

# --- Phase 2: Resume from checkpoint and provide input ---
print("\n📌 Phase 2: Resuming from checkpoint...")
checkpoints = await storage.list_checkpoints()
if checkpoints:
    latest = sorted(checkpoints, key=lambda c: c.timestamp, reverse=True)[0]
    print(f"   Restoring checkpoint: {latest.checkpoint_id}")

    # Restore checkpoint to reload pending requests
    restored_result = await workflow_durable.run(checkpoint_id=latest.checkpoint_id)
    restored_requests = restored_result.get_request_info_events()

    # Send responses
    responses: dict[str, object] = {}
    for req in restored_requests:
        if isinstance(req.data, UserInputRequest):
            responses[req.request_id] = "Yes, please process the refund for order 12345."
            print(f"   📝 Responded to agent: {req.data.agent_name}")

    if responses:
        final_result = await workflow_durable.run(responses=responses)
        if final_result.get_outputs():
            print("\n✅ Durable workflow completed after checkpoint restore!")
            print(final_result.get_outputs()[0])
else:
    print("   No checkpoints found.")

## 📝 Key Takeaways

### Handoff Orchestration for Customer Support

| Benefit | Description |
|---------|-------------|
| **Dynamic Routing** | Agents decide which specialist handles the next interaction |
| **Specialized Expertise** | Each agent focuses on their domain |
| **Context Preservation** | Full conversation history maintained across handoffs |
| **Flexible Routing** | Default mesh or custom handoff rules via `add_handoff()` |
| **Autonomous Mode** | Optional mode for unattended workflows with turn limits |
| **Tool Approval (HITL)** | Human approval for sensitive operations |
| **Checkpointing** | Durable workflows that survive process restarts |

### When to Use Handoff vs Other Patterns

| Pattern | When to Use | Key Feature |
|---------|-------------|-------------|
| **Handoff** | Dynamic delegation between peers | Agents transfer full ownership |
| **Sequential** | Linear, dependent steps | Fixed execution order |
| **Magentic** | Complex, adaptive decomposition | Central orchestrator manages plan |
| **Agent-as-Tools** | Primary agent delegates subtasks | Primary retains control |

### Key APIs Reference

| API | Purpose |
|-----|---------|
| `HandoffBuilder` | Build handoff workflows |
| `.with_start_agent()` | Set the entry-point agent |
| `.add_handoff()` | Configure custom routing rules |
| `.with_autonomous_mode()` | Enable autonomous operation |
| `HandoffAgentUserRequest` | Handle user input events |
| `FunctionApprovalRequestContent` | Handle tool approval events |
| `FileCheckpointStorage` | Enable durable checkpointing |
| `workflow.run()` / `workflow.send_responses()` | Non-streaming workflow execution |
| `WorkflowRunResult.get_request_info_events()` | Get pending requests from result |
| `@ai_function(approval_mode="always_require")` | Mark tools requiring approval |

### Production Considerations for FSI

| Consideration | Recommendation |
|---------------|----------------|
| **Routing Rules** | Use `add_handoff()` for controlled routing in regulated environments |
| **Audit Trail** | Log all handoff decisions and agent interactions |
| **Autonomous Mode** | Use sparingly; set `turn_limits` to prevent runaway workflows |
| **Tool Approval** | Use `@ai_function(approval_mode="always_require")` for sensitive operations |
| **Checkpointing** | Enable for long-running workflows that may span sessions |
| **Context Sync** | Remember: only user/agent messages are synchronized, not tool calls |